# Deep JSCC — Tutorial Demo

End-to-end Deep Joint Source-Channel Coding on CIFAR-10, with two stories:

1. **Robust SNR training** beats fixed-SNR training.
2. **Rate-distortion trade-off** — how much bandwidth `k/n` do you actually need?

**Before running:** *Runtime → Change runtime type → T4 GPU* (free).

## 0. Setup

In [ ]:
!git clone https://github.com/samerlahoud/deep-jscc-demo.git
%cd deep-jscc-demo
!pip install -q -r requirements.txt

## 1. Story 1 — Robust SNR training

Train two fixed-SNR models (`SNR_train ∈ {1, 7}` dB) and one robust model over the full operating range `SNR ~ U(0, 20)`. ~10 min on T4.

In [ ]:
!python train.py                          # fixed SNRs: {1, 7} dB
!python train.py --snr_train_range 0 20   # robust model

In [ ]:
!python eval.py                           # PSNR/SSIM curves + reconstruction grid

In [ ]:
from IPython.display import Image, display
display(Image('results/awgn_snr_sweep.png'))
display(Image('results/awgn_reconstructions.png'))

**Reading the figures:**
- The robust curve is bold. It sits at or above the fixed curves across the whole test range — that's the headline.
- The reconstruction grid alternates **Raw** (pixels through the channel, no encoder) and **JSCC**. At 1 dB the Raw column is pure snow; JSCC still gives a recognizable image. This is the elevator pitch for semantic communication.

## 2. Story 2 — Rate-distortion: how much bandwidth do we need?

Train at a single SNR (7 dB) across several latent sizes — i.e. several **bandwidth ratios `k/n`**. Then plot quality vs. bandwidth.

| `latent_ch` |  k/n  |
|------------:|:-----:|
|           4 | 1/12  |
|           8 | 1/6   |
|          16 | 1/3   |
|          24 | 1/2   |

In [ ]:
!python train.py --snr_train_list 7 --latent_ch_list 4 8 16 24

In [ ]:
!python eval.py --mode bw_sweep --bw_train_snr 7dB

In [ ]:
display(Image('results/awgn_bw_sweep.png'))
display(Image('results/awgn_bw_reconstructions.png'))

**Reading the figures:**
- Quality saturates as `k/n` grows — diminishing returns. The knee tells you the cheapest bandwidth you can get away with.
- The reconstruction grid shows the same images at each bandwidth. The drop in detail from `k/n=1/2` to `k/n=1/12` is visible.

## 3. Optional — Rayleigh fading

Repeat Story 1 with a Rayleigh channel instead of AWGN.

In [ ]:
# !python train.py --channel rayleigh
# !python train.py --channel rayleigh --snr_train_range 0 20
# !python eval.py  --channel rayleigh
# display(Image('results/rayleigh_snr_sweep.png'))